## Multi-arm Environment with mjSpec
- spawn multiple UR5e using mjSpec

In [1]:
import mujoco as mj
import mujoco_viewer

import numpy as np
import time

import xml.etree.ElementTree as ET
from lxml import etree

In [2]:
def print_xml(xml_input,color=True):
    if isinstance(xml_input, ET.Element):
        rough_string = ET.tostring(xml_input, encoding='unicode')
    else:
        rough_string = xml_input

    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.fromstring(rough_string, parser=parser)
    pretty_xml = etree.tostring(tree, pretty_print=True, encoding='unicode')
    print(pretty_xml)

### Synthesize mjSpec scene
- load default floor
- add frame & attach UR robot

In [3]:
path = '../floor_white_gray.xml'
spec = mj.MjSpec.from_file(path)

In [4]:
multi_robot_config = {
    "frame": {
        "frame1": {
            "name": "UR_1",
            "pos": [1, 0, 0],
            "quat": [0.7071, 0, 0, 0.7071]
        },
        "frame2": {
            "name": "UR_2",
            "pos": [0, 1, 0],
            "quat": [0, 0, 0, 1]
        },
        "frame3": {
            "name": "UR_3",
            "pos": [-1, 0, 0],
            "quat": [0.7071, 0, 0, -0.7071]
        },
        "frame4": {
            "name": "UR_4",
            "pos": [0, -1, 0],
            "quat": [1, 0, 0, 0]
        }
    },
    
    "spec_file": "../ur5e_mjcf/ur5e.xml"
}

In [ ]:
# attach frames
frame_list = []
for key, info in multi_robot_config["frame"].items():
    frame = spec.worldbody.add_frame(pos=info["pos"], quat=info["quat"])
    frame_list.append(frame)

In [ ]:
# load robot spec
for i, frame in enumerate(frame_list):
    spec_arm = mj.MjSpec.from_file("../ur5e_mjcf/ur5e.xml")
    frame.attach_body(spec_arm.body("base"), 'attached-', f'-{i}')

In [8]:
model = spec.compile()
data = mj.MjData(model)

In [ ]:
""" MAIN LOOP """

# create python viewer object
viewer = mj_viewer.MujocoViewer(model, data)

# data reset
mj.mj_resetData(model, data)

while True:
    if viewer.is_alive:
        
        mj.mj_step(model, data)
        viewer.render()

    else:
        break

# close
viewer.close()

Pressed ESC
Quitting.


In [10]:
print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <size nkey="4"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <default>
    <default class="attached-main-0">
      <default class="attached-ur5e-0">
        <material shininess="0.25"/>
        <joint range="-6.28319 6.28319" armature="0.1"/>
        <site size="0.001 0.005 0.005" group="4" rgba="0.5 0.5 0.5 0.3"/>
        <general ctrlrange="-6.2831 6.2831" forcerange="-150 150" biastype="affine" gainprm="2000" biasprm="0 -2000 -400"/>
        <default class="attached-size3-0">
          <default class="attached-size3_limited-0">
            <joint range="-3.1415 3.1415"/>
            <general ctrlrange="-3.1415 3.1415"/>
          </default>
        </default>
        <default class="attached-size1-0">
          <general forcerange="-28 28" gainprm="500" biasprm="0 -500 -100"/>
        </default>
        <default class="attached-visual-0">
     

In [ ]:
def get_actuator_name (model, data):
    control_names = [mj.mj_id2name(model,mj.mjtObj.mjOBJ_ACTUATOR,ctrl_idx) for ctrl_idx in range(model.nu)]
    return control_names

actuator_names = get_actuator_name(model,data) # same naming convention - prefix and postfix

In [15]:
model.nu

24